# Phase

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389.
Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
from collections import Counter
from math import pi

import torch
from IPython.display import Markdown, SVG, display

from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum import Circuit
from flagquantum.compiler._component_trace.call_graph import modeled_call_graph
from flagquantum.compiler._component_trace.resource_model import (
    ComponentCost, GateCost, estimate_component_resources,
)
from flagquantum.core.ir import Instruction
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import (
    export_modeled_call_graph_dot, format_call_graph_markdown, render_call_graph_svg,
)
from flagquantum.simulation.unitary import get_unitary

## Phase of a block encoding

For a block encoding of $A$, a scalar phase produces a block encoding of
$\exp(i\pi\phi)A$. The wrapper keeps the child normalization, ordered system
ports, any ancilla and resource ports, and the signal preparation. The error
metadata is the child error plus the supplied phase precision.

In [ ]:
from flagquantum.algorithms.primitives.phase_block_encoding import PhaseBlockEncoding
from flagquantum.algorithms.primitives.unitary_block_encoding import UnitaryBlockEncoding

### Example Instances

In [ ]:
def hadamard(ports, workspace, classical_bits, adjoint):
    return (Instruction("h", ports["q"]),)

child = UnitaryBlockEncoding((("q", 1),), hadamard)
phase_block = PhaseBlockEncoding(child, phi=0.25, eps=0)
assert (phase_block.alpha, phase_block.epsilon) == (1, 0)

#### Graphical Signature

In [ ]:
signature = Signature.build(system=phase_block.system_bitsize)
display(SVG(field_signature_svg([("Phase of Hadamard", signature, {})])))

### Call Graph

In [ ]:
model = phase_block.resource_model(ComponentCost("Hadamard", 1, GateCost(clifford=1)))
graph = modeled_call_graph(model, max_depth=1)
display(SVG(render_call_graph_svg(export_modeled_call_graph_dot(model, max_depth=1))))
display(Markdown(format_call_graph_markdown(
    graph.root, graph.direct_calls, graph.leaf_counts, title=lambda node: node.name,
)))

### Verify the absolute phase

The source models a zero-wire scalar phase. The physical append path uses
an existing anchor. For this fixture the system wire is the anchor. The two
phase instructions and two X gates preserve that wire's state while applying
the scalar to the entire circuit. The independent source model charges zero
for the scalar and one Clifford for Hadamard.

In [ ]:
circuit = Circuit(1, dtype=torch.complex128)
circuit.append(phase_block.make_on(child.make_on(system=(0,)), anchor=0))
h = torch.tensor([[1, 1], [1, -1]], dtype=torch.complex128) / 2**0.5
theta = torch.tensor(0.25j * pi, dtype=torch.complex128)
torch.testing.assert_close(get_unitary(circuit), theta.exp() * h, atol=2e-12, rtol=0)
print("Emitted primitives", dict(Counter(gate.name for gate in circuit.to_ir().instructions)))
print("Source resource model", estimate_component_resources(model))
circuit.append(phase_block.make_on(
    child.make_on(system=(0,), adjoint=True), anchor=0, adjoint=True,
))
torch.testing.assert_close(get_unitary(circuit), torch.eye(2, dtype=torch.complex128),
                           atol=2e-12, rtol=0)